In [ ]:
#| default_exp clustering.core

# core

> This module contains the implementation of the Gaussian Mixture Model (GMM)

In [1]:
#| hide
from nbdev import show_doc
from nbdev.showdoc import *

In [2]:
#| export
#| hide

import jax
import jax.numpy as jnp
from jax.scipy.special import logsumexp
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import KFold
import numpy as np

## GMM

In [7]:
#| export
#| hide


class GaussianMixture:
    def __init__(self, n_components, tol=1e-6, max_iter=100,
                 reg_covar=1e-6, init_params='kmeans'):
        """
            Gaussian Mixture Model with EM algorithm and K-means initialization.
            
            Args:
            
                n_components (int): Number of Gaussian components.
                tol (float): Convergence tolerance.
                max_iter (int): Maximum number of EM iterations.
                reg_covar (float): Regularization for covariance matrix.
                init_params (str): Initialization method ('kmeans' or 'random').
        """
        
        self.n_components = n_components
        self.tol = tol
        self.max_iter = max_iter
        self.reg_covar = reg_covar
        self.init_params = init_params
        # learned parameters
        self.means_ = None        # (K, D)
        self.covariances_ = None  # (K, D, D)
        self.weights_ = None      # (K,)

    def _init_params(self, X, key):
        N, D = X.shape
        # default uniform weights
        weights = jnp.ones(self.n_components) / self.n_components

        # init means via k-means or random selection
        if self.init_params == 'kmeans':
            # sklearn KMeans on CPU
            kmeans = KMeans(n_clusters=self.n_components, n_init=10)
            labels = kmeans.fit_predict(jax.device_get(X))
            means = jnp.array(kmeans.cluster_centers_)
            # set weights based on cluster sizes
            counts = jnp.bincount(jnp.array(labels), length=self.n_components)
            weights = counts / N
        else:
            perm = jax.random.permutation(key, N)
            means = X[perm[:self.n_components]]

        # shared covariance estimate
        emp_cov = jnp.cov(X, rowvar=False) + self.reg_covar * jnp.eye(D)
        covariances = jnp.tile(emp_cov, (self.n_components, 1, 1))
        return weights, means, covariances

    def _estimate_log_gaussian_prob(self, X, means, covariances):
        N, D = X.shape
        def log_prob_k(mu, cov):
            diff = X - mu
            sign, logdet = jnp.linalg.slogdet(cov)
            inv_cov = jnp.linalg.inv(cov)
            m_dist = jnp.sum(diff @ inv_cov * diff, axis=1)
            return -0.5 * (D * jnp.log(2 * jnp.pi) + logdet + m_dist)
        log_probs = jax.vmap(log_prob_k, in_axes=(0, 0))(means, covariances)
        return log_probs.T  # (N, K)



    def fit(self, X, key=jax.random.PRNGKey(0)):
        """
        Fit the Gaussian mixture model using EM.

        X : array (N, D)
        key : PRNGKey for random init
        """
        weights, means, covariances = self._init_params(X, key)
        lower_bound = -jnp.inf


        @jax.jit
        def _e_step(X, weights, means, covariances):
            log_gauss = self._estimate_log_gaussian_prob(X, means, covariances)
            log_weights = jnp.log(weights)
            log_prob = log_gauss + log_weights
            log_prob_norm = logsumexp(log_prob, axis=1, keepdims=True)
            log_resp = log_prob - log_prob_norm
            resp = jnp.exp(log_resp)
            return resp, jnp.squeeze(log_prob_norm)

        @jax.jit
        def _m_step(X, resp):
            N, D = X.shape
            Nk = jnp.sum(resp, axis=0) + 1e-10
            weights = Nk / N
            means = (resp.T @ X) / Nk[:, None]
            def cov_k(r, mu):
                diff = X - mu
                weighted = r[:, None] * diff
                cov = (weighted.T @ diff) / (jnp.sum(r) + 1e-10)
                return cov + self.reg_covar * jnp.eye(D)
            covariances = jax.vmap(cov_k, in_axes=(1, 0))(resp, means)
            return weights, means, covariances


        for _ in range(self.max_iter):
            resp, log_prob_norm = _e_step(X, weights, means, covariances)
            curr_lb = jnp.mean(log_prob_norm)
            if jnp.abs(curr_lb - lower_bound) < self.tol:
                break
            lower_bound = curr_lb
            weights, means, covariances = _m_step(X, resp)

        self.weights_, self.means_, self.covariances_ = weights, means, covariances
        return self

    #@jax.jit
    def predict_proba(self, X):
        log_gauss = self._estimate_log_gaussian_prob(X, self.means_, self.covariances_)
        log_prob = log_gauss + jnp.log(self.weights_)
        log_prob_norm = logsumexp(log_prob, axis=1, keepdims=True)
        return jnp.exp(log_prob - log_prob_norm)

    #@jax.jit
    def predict(self, X):
        return jnp.argmax(self.predict_proba(X), axis=1)


    def _n_parameters(self, D):
        """Number of free parameters for full-covariance GMM:
           weights: K-1, means: K*D, covariances: K * D*(D+1)/2
        """
        K = self.n_components
        cov_params = K * D * (D + 1) // 2
        mean_params = K * D
        weight_params = K - 1
        return int(weight_params + mean_params + cov_params)

    def score_samples(self, X):
        """Return per-sample log-likelihood under current model (array shape (N,))."""
        # log N(x) = logsumexp_k [ log w_k + log N(x|mu_k, cov_k) ]
        log_gauss = self._estimate_log_gaussian_prob(X, self.means_, self.covariances_)  # (N, K)
        log_weights = jnp.log(self.weights_)
        log_prob = log_gauss + log_weights  # (N, K)
        log_prob_norm = logsumexp(log_prob, axis=1)  # (N,)
        return log_prob_norm

    def score(self, X, average=True):
        """Return total (or average) log-likelihood of X under model.
           average=True -> average log-likelihood per sample (float)
           average=False -> total log-likelihood (float)
        """
        log_prob_norm = self.score_samples(X)
        if average:
            return jnp.mean(log_prob_norm)
        else:
            return jnp.sum(log_prob_norm)

    def aic(self, X):
        """AIC = 2*p - 2*logL (we compute using total log-likelihood)"""
        N, D = X.shape
        p = self._n_parameters(D)
        ll = float(self.score(X, average=False))
        return 2 * p - 2 * ll

    def bic(self, X):
        """BIC = p*log(N) - 2*logL"""
        N, D = X.shape
        p = self._n_parameters(D)
        ll = float(self.score(X, average=False))
        return p * jnp.log(N) - 2 * ll

    def responsibilities_entropy(self, X):
        """Average entropy of posterior responsibilities (measure of overlap/uncertainty)."""
        # E-step (reuse existing helper logic)
        log_gauss = self._estimate_log_gaussian_prob(X, self.means_, self.covariances_)
        log_weights = jnp.log(self.weights_)
        log_prob = log_gauss + log_weights
        log_prob_norm = logsumexp(log_prob, axis=1, keepdims=True)
        log_resp = log_prob - log_prob_norm
        resp = jnp.exp(log_resp)  # (N, K)
        # entropy per sample: -sum_k r_k log r_k
        entropy = -jnp.sum(resp * jnp.where(resp > 0, jnp.log(resp), 0.0), axis=1)
        return jnp.mean(entropy)

    def evaluate_internal(self, X):
        """Convenience: return dict of internal metrics (AIC, BIC, avg logL, entropy)."""
        return {
            "aic": float(self.aic(X)),
            "bic": float(self.bic(X)),
            "avg_log_likelihood": float(self.score(X, average=True)),
            "avg_resp_entropy": float(self.responsibilities_entropy(X)),
        }

def select_n_components(X, ks, n_init=5, max_iter=100, covariant_reg=1e-6, init_params='kmeans', random_seed=0):
    """
    Try multiple K values and several initializations, return a summary dict.

    Returns:
      results: dict keyed by K containing the best model and metrics (bic, aic, avg_log_lik, silhouette if possible).
    """
    results = {}
    key = jax.random.PRNGKey(random_seed)
    for K in ks:
        print(f"Trying K={K}")
        best_model = None
        best_ll = -np.inf
        best_metrics = None
        for i in range(n_init):
            key, subkey = jax.random.split(key)
            gmm = GaussianMixture(n_components=K, max_iter=max_iter, reg_covar=covariant_reg, init_params=init_params)
            #try:
            gmm.fit(X, key=subkey)
            #except Exception as e:
                # EM sometimes fails numerically; skip this init
             #   continue
            avg_ll = float(gmm.score(X, average=True))
            if avg_ll > best_ll:
                best_ll = avg_ll
                best_model = gmm
                best_metrics = gmm.evaluate_internal(X)
                best_metrics['silhouette'] = np.nan
        if best_model is not None:
            results[K] = {"model": best_model, "metrics": best_metrics}
        else:
            results[K] = {"model": None, "metrics": None}
    return results



In [8]:
show_doc(GaussianMixture.__init__, name='GaussianMixture.init')

---

### GaussianMixture.init

```python

def __init__(
    n_components, tol:float=1e-06, max_iter:int=100, reg_covar:float=1e-06, init_params:str='kmeans'
):


```

*Gaussian Mixture Model with EM algorithm and K-means initialization.*

Args:

    n_components (int): Number of Gaussian components.
    tol (float): Convergence tolerance.
    max_iter (int): Maximum number of EM iterations.
    reg_covar (float): Regularization for covariance matrix.
    init_params (str): Initialization method ('kmeans' or 'random').

In [9]:
show_doc(GaussianMixture.fit, name='GaussianMixture.fit')

---

### GaussianMixture.fit

```python

def fit(
    X, key:ArrayImpl=Array([0, 0], dtype=uint32)
):


```

*Fit the Gaussian mixture model using EM.*

X : array (N, D)
key : PRNGKey for random init

# export

In [30]:
#| hide
import nbdev; nbdev.nbdev_export()